In [ ]:
# 6차시 보강 실습: 손실함수 비교 (Colab 호환)
import torch, torch.nn as nn, torch.optim as optim
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)

# 활성화 함수와 손실 함수는 딥러닝 모델이 학습하고 예측하는 과정에서 서로 다른 역할을 수행하는 핵심 요소입니다.
# 두 함수의 차이점
# 활성화 함수(Activation Function): 모델의 내부(은닉층)나 출력층에서 입력받은 데이터를 변환하여 
# 다음 층으로 전달할지 말지, 전달한다면 어떤 형태로 변환할지 결정합니다. 주로 신경망에 비선형성을 부여하여 
# 모델이 단순한 직선 형태가 아닌 복잡한 패턴을 학습할 수 있게 돕습니다.
# 시그모이드, 렐루, 소프트맥스, 하이퍼볼릭 탄젠트 등

# 손실 함수(Loss Function): 모델의 최종 출력(예측값)과 실제 정답(Label) 간의 차이(오차)를 수치화하여
# 평가하는 함수입니다. 모델은 이 오차(손실)를 최소화하는 방향으로 파라미터(가중치)를 스스로 업데이트하며 학습합니다.
# 요약하자면 활성화 함수는 "데이터를 어떻게 가공해서 다음으로 넘길 것인가"를 결정하고, 
# 손실 함수는 "모델의 예측이 정답과 얼마나 틀렸는가"를 평가하는 채점 기준입니다.
# 1. 회귀(Regression) 문제의 손실 함수
# 회귀는 예측값이 연속적인 숫자(예: 집값, 온도 등)일 때 사용합니다.
# MSE (Mean Squared Error, 평균 제곱 오차): 가장 기본적이고 널리 쓰이는 손실 함수입니다. 
# 예측값과 실제값의 차이를 제곱한 뒤 평균을 냅니다. 오차를 제곱하기 때문에 정답에서 크게 벗어난 
# 이상치(Outlier)에 민감하게 반응하여 패널티를 강하게 부여합니다.
# MAE (Mean Absolute Error, 평균 절대 오차): 예측값과 실제값의 차이에 절댓값을 씌워 평균을 냅니다.
# MSE에 비해 이상치(Outlier)에 덜 민감하다는 장점이 있습니다.
# RMSE (Root Mean Squared Error, 평균 제곱근 오차): MSE 값에 루트(제곱근)를 씌운 것입니다.
# 오차의 단위를 실제 데이터의 단위와 맞춰주어 해석이 직관적입니다.
# Huber Loss (후버 손실): MSE와 MAE의 장점을 결합한 함수입니다. 오차가 작을 때는 MSE처럼 작동하여 
# 정밀하게 학습하고, 오차가 클 때는 MAE처럼 작동하여 이상치에 덜 민감하게 대처합니다.
# 2. 분류(Classification) 문제의 손실 함수
# 분류는 데이터가 어떤 카테고리(클래스)에 속하는지 예측할 때 사용합니다.
# BCE (Binary Cross-Entropy Loss, 이진 교차 엔트로피): 결과가 두 가지(예: 스팸이다/아니다, 0 또는 1) 
# 중 하나로 나오는 이진 분류 모델에서 주로 사용합니다. 일반적으로 출력층의 시그모이드(Sigmoid) 활성화 함수와 짝을 이루어 사용됩니다.
# CCE (Categorical Cross-Entropy Loss, 범주형 교차 엔트로피): 
# 결과가 세 개 이상의 카테고리(예: 개, 고양이, 토끼 중 하나)인 다중 클래스 분류 문제에서 널리 사용됩니다. 
# 정답 라벨이 원-핫 인코딩(One-hot encoding) 형태로 되어 있을 때 사용하며, 
# 모델의 출력층에서 소프트맥스(Softmax) 함수와 결합하여 사용됩니다.
# SCCE (Sparse Categorical Cross-Entropy Loss): 범주형 교차 엔트로피와 역할은 같지만, 
# 정답 라벨이 원-핫 인코딩(예: [0, 1, 0])이 아니라 정수형 인덱스(예: 1)로 되어 있을 때 사용합니다.
# Hinge Loss (힌지 손실): 주로 서포트 벡터 머신(SVM) 알고리즘에서 이진 분류를 할 때 사용하는 손실 함수입니다. 
# 정답을 맞춘 경우에는 손실을 0으로 처리하고, 예측이 틀렸거나 경계선에 너무 가까운 경우에만 패널티를 부여합니다.

# ============================================================
# 손실함수 비교 실습
# (A) 이진분류: MSE vs BCEWithLogitsLoss
# (B) 다중분류: Label Smoothing vs Class Weights
# ============================================================

# ============================================================
# (A) Binary Classification: MSE vs BCEWithLogitsLoss
# ============================================================

# ------------------------------------------------------------
# 2. 이진분류용 가상 데이터 생성
# ------------------------------------------------------------
# make_classification(...)
# - sklearn에서 분류용 샘플 데이터를 쉽게 만들어주는 함수임
# - n_samples=4000      : 총 데이터 개수 4000개
# - n_features=20       : 입력 특성(feature) 20개
# - n_informative=8     : 실제로 분류에 의미 있는 feature는 8개
# - weights=[0.6, 0.4]  : 클래스 비율이 60:40 되도록 생성
# - random_state=0      : 재현성 위해 고정
X, y = make_classification(
    n_samples=4000,
    n_features=20,
    n_informative=8,
    weights=[0.6, 0.4],
    random_state=0
)

# ------------------------------------------------------------
# 3. 학습/테스트 데이터 분리 + torch 텐서 변환
# ------------------------------------------------------------
# X는 입력 데이터
# y는 정답 라벨(0 또는 1)
#
# torch.tensor(..., dtype=torch.float32)
# - numpy 배열을 파이토치 텐서로 바꿈
# - float32로 맞춰서 신경망 계산에 쓰기 쉽게 만듦
#
# y는 unsqueeze(1)로 [N] -> [N, 1]로 바꿈
# - 최종 출력층이 1개짜리 binary classifier라서 shape 맞추는 용도임
# - 예: [2800] -> [2800, 1]
Xtr, Xte, ytr, yte = train_test_split(
    torch.tensor(X, dtype=torch.float32),
    torch.tensor(y, dtype=torch.float32).unsqueeze(1),
    test_size=0.3,
    random_state=0
)


# ------------------------------------------------------------
# 4. 이진분류용 간단한 MLP 모델 정의
# ------------------------------------------------------------
class BinNet(nn.Module):
    def __init__(self):
        super().__init__()

        # 20차원 입력 -> 64차원 은닉층 -> ReLU -> 1차원 출력
        #
        # 마지막 출력층에 sigmoid를 넣지 않은 점이 중요함
        # 이유:
        # - BCEWithLogitsLoss를 쓸 때는 raw output(logit)을 그대로 넣어야 함
        # - sigmoid는 손실함수 내부에서 처리함
        self.m = nn.Sequential(
            nn.Linear(20, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.m(x)


# ------------------------------------------------------------
# 5. 이진분류 학습/평가 함수
# ------------------------------------------------------------
def run_binary(loss_fn):
    # 새 모델 생성 후 device로 이동
    net = BinNet().to(device)

    # AdamW 옵티마이저 사용
    # lr=1e-3 : 학습률 0.001
    opt = optim.AdamW(net.parameters(), lr=1e-3)
    # 자세한 사용 시기와 이유는 다음과 같습니다.

    # 1. 일반화(Generalization) 성능을 높이고 싶을 때
    # 기존의 Adam은 학습 속도가 빠르고 수렴이 잘 되어 만능처럼 여겨졌지만, 
    # 특정 문제(특히 컴퓨터 비전 분야 등)에서는 SGD(확률적 경사 하강법) 
    # 모델보다 새로운 데이터에 대한 예측 성능(일반화 성능)이 떨어진다는 단점이 있었습니다. 
    # AdamW는 이러한 Adam의 약점을 보완하여 일반화 성능을 크게 끌어올린 모델이므로, 
    # 학습 데이터 외에 새로운 데이터에서도 모델이 잘 작동하게 만들고 싶을 때 사용합니다.

    # 2. 가중치 감쇠(Weight Decay)를 효과적으로 적용하고 싶을 때
    # 딥러닝에서 모델이 너무 복잡해져서 훈련 데이터에만 과하게 맞춰지는 과적합을 막기 위해 
    # 'L2 정규화(Weight Decay)'라는 기법을 사용합니다.

    # 그런데 기존 Adam은 가중치 감쇠를 기울기(Gradient) 계산 과정에 섞어서 처리하다 보니, 
    # Adam 특유의 '파라미터별 적응적 학습률(Adaptive Learning Rate)' 기능과 충돌하여 정규화 효과가 제대로 발휘되지 못했습니다.

    # 반면 AdamW는 이 가중치 감쇠 과정을 완전히 분리(Decoupled)하여 독립적으로 적용함으로써 
    # 오버피팅을 훨씬 더 안정적으로 완화합니다.

    # 3. 최신 트랜스포머(Transformer) 기반 모델을 학습할 때
    # 최근 나오는 대규모 언어 모델(BERT, GPT 등)이나 비전 모델(Vision Transformer, YOLO의 최신 버전 등)은 
    # 모델의 크기가 커서 과적합 관리가 매우 중요합니다. 이런 이유로 최근 Hugging Face 등을 활용한 
    # 최신 딥러닝 실무에서는 거의 기본값 수준으로 Adam 대신 AdamW를 채택하는 추세입니다.



    # epoch 8번만 간단히 학습
    for _ in range(8):
        # 학습 모드 전환
        net.train()

        # 이전 step gradient 초기화
        opt.zero_grad()

        # 순전파
        # Xtr 전체를 한 번에 넣어서 출력 계산
        # out shape은 [N, 1]
        #
        # 여기 out은 "확률"이 아니라 raw score(logit)임
        out = net(Xtr.to(device))
        # 손실 계산
        # loss_fn 자리에 어떤 손실함수를 넣느냐에 따라 비교 실험 진행됨
        # 주의할 점: PyTorch에서는 모델과 데이터가 반드시 같은 장치(device) 위에 있어야 연산이 가능합니다. 
        # 하나는 CPU에 있고 하나는 GPU에 있으면 에러가 발생합니다.
        loss = loss_fn(out, ytr.to(device))

        # 역전파
        loss.backward()

        # 파라미터 업데이트
        opt.step()

    # 평가 단계
    # gradient 필요 없으므로 no_grad 사용
    #     with torch.no_grad():는 PyTorch에서 기울기(Gradient) 계산을 비활성화할 때 
    # 사용하는 컨텍스트 매니저(Context Manager)입니다.

    # PyTorch는 기본적으로 텐서(Tensor)들의 연산 기록을 모두 추적하여 나중에 backward()를 통해 
    # 역전파(Backpropagation)와 가중치 업데이트를 할 수 있도록 준비합니다. 하지만 with torch.no_grad(): 
    # 블록 안에서 실행되는 코드들은 이 연산 추적(Autograd 엔진)이 꺼지게 됩니다.

    # 주로 언제 쓰는가?
    # 크게 두 가지 상황에서 사용합니다.

    # 모델 평가(Validation/Testing) 및 추론(Inference) 시
    # 학습이 끝난 모델로 결과를 예측만 할 때는 역전파를 통해 가중치를 업데이트할 필요가 없습니다. 
    # 이때 with torch.no_grad():를 사용하면 불필요한 기울기 계산과 연산 기록 저장을 막아주어 
    # 메모리 사용량을 크게 줄이고 연산 속도를 높일 수 있습니다.

    # 수동으로 가중치를 업데이트할 때
    # 사용자 지정 옵티마이저를 구현하거나 가중치(Weight)를 직접 초기화하고 수정할 때 사용합니다. 
    # 가중치를 수정하는 행위 자체가 연산으로 기록되어 또 다른 기울기를 만들어내는 것을 방지하기 위함입니다. 
    # (예: W -= lr * W.grad와 같은 코드 )
    with torch.no_grad():
        # 테스트셋에 대한 raw output(logit) 계산
        # 그 다음 sigmoid를 적용해서 0~1 확률로 변환
        p = torch.sigmoid(net(Xte.to(device)))

        # p > 0.5 이면 1, 아니면 0으로 예측
        # float()로 0.0 / 1.0 변환
        #
        # yte와 비교해서 맞으면 True, 틀리면 False
        # 다시 float() 후 평균내면 정확도(acc) 됨
        acc = ((p > 0.5).float() == yte.to(device)).float().mean().item()

    return acc


# ------------------------------------------------------------
# 6. MSE로 이진분류 학습
# ------------------------------------------------------------
# 주의할 점:
# - MSELoss는 원래 회귀에서 많이 쓰는 손실함수임
# - 이진분류에서도 "확률값"과 정답(0/1)의 제곱오차로 쓸 수는 있음
# - 그래서 여기서는 out에 sigmoid를 먼저 적용한 뒤 MSE를 계산함
#
# 즉 흐름은
# raw logit -> sigmoid -> 확률 -> MSE 계산
acc_mse = run_binary(
    lambda out, y: nn.MSELoss()(torch.sigmoid(out), y)
)

# ------------------------------------------------------------
# 7. BCEWithLogitsLoss로 이진분류 학습
# ------------------------------------------------------------
# BCEWithLogitsLoss는
# - sigmoid + binary cross entropy를 합쳐 놓은 손실함수임
# - raw logit(out)을 그대로 넣으면 내부에서 안정적으로 처리함
#
# 그래서 이 경우에는
# loss 계산할 때 따로 sigmoid를 하지 않음
acc_bce = run_binary(nn.BCEWithLogitsLoss())

# 최종 정확도 비교 출력
print(f"Binary Acc: MSE={acc_mse:.3f} vs BCEWithLogits={acc_bce:.3f}")


# ============================================================
# (B) Multiclass Classification: Label Smoothing & Class Weights
# ============================================================
# 1. 왜 필요할까? (The Problem)보통 분류 문제에서는 정답은 1, 오답은 0으로 표시하는 원-핫 인코딩(One-hot Encoding)을 
# 사용해.예: [개, 고양이, 사자] 중 고양이가 정답이면 [0, 1, 0]그런데 모델이 이 1을 맞추기 위해 극단적으로 학습하다 보면, 
# 출력값(Logits)이 무한대로 커지려는 경향이 생겨. 이걸 과잉 확신(Over-confidence)이라고 하는데, 
# 이게 심해지면 조금만 데이터가 달라도 틀려버리는 과적합(Overfitting)에 빠지기 쉬워져.
# 2. 어떻게 작동할까? (The Mechanism)라벨 스무딩은 정답의 1에서 아주 조금($\alpha$)을 떼어내서 
# 나머지 오답들에게 골고루 나눠줘.만약 $\alpha$가 0.1이라면, 정답 라벨은 0.9가 되고 나머지 0.1은 클래스 개수($K$)
# 만큼 나눠서 배분해 # 원-핫 라벨: [0, 1, 0]라벨 스무딩 적용 (예): [0.033, 0.934, 0.033]
# ------------------------------------------------------------
# 8. MNIST 전처리 정의
# ------------------------------------------------------------
# transforms.ToTensor()
# - 이미지를 torch.Tensor로 바꿈
# - 보통 픽셀값 0~255를 0~1 범위 float로 바꿔줌
#
# Compose는 여러 전처리를 묶는 컨테이너임
tfm = transforms.Compose([
    transforms.ToTensor()
])

# ------------------------------------------------------------
# 9. MNIST 데이터셋 로드
# ------------------------------------------------------------
# train=True  : 학습셋 60,000장
# train=False : 테스트셋 10,000장
# download=True : 없으면 다운로드
# transform=tfm : 이미지 -> 텐서 변환
train_ds = datasets.MNIST('/tmp/mnist2', train=True,  download=True, transform=tfm)
test_ds  = datasets.MNIST('/tmp/mnist2', train=False, download=True, transform=tfm)

# ------------------------------------------------------------
# 10. DataLoader 생성
# ------------------------------------------------------------
# tr
# - batch_size=256 : 학습 시 256장씩 사용
# - shuffle=True   : 학습 데이터는 섞는 게 일반적임
#
# te
# - batch_size=512 : 평가 시 좀 더 크게 잡아도 괜찮음
# - shuffle=False  : 테스트는 순서 안 섞어도 됨
tr = DataLoader(train_ds, batch_size=256, shuffle=True)
te = DataLoader(test_ds,  batch_size=512, shuffle=False)


# ------------------------------------------------------------
# 11. 작은 CNN 모델 정의
# ------------------------------------------------------------
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            # -----------------------------
            # [특징 추출부]
            # -----------------------------
            # 입력 이미지 shape: [B, 1, 28, 28]
            #
            # Conv2d(1,16,3,padding=1)
            # - 입력 채널 1개(흑백)
            # - 출력 채널 16개
            # - 커널 크기 3x3
            # - padding=1이라 공간 크기 유지
            # 결과 shape: [B, 16, 28, 28]
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),

            # MaxPool2d(2)
            # - 가로세로 절반으로 줄임
            # 결과 shape: [B, 16, 14, 14]
            nn.MaxPool2d(2),

            # 두 번째 합성곱층
            # [B, 16, 14, 14] -> [B, 32, 14, 14]
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),

            # 다시 pooling
            # [B, 32, 14, 14] -> [B, 32, 7, 7]
            nn.MaxPool2d(2),

            # -----------------------------
            # [분류기 부분]
            # -----------------------------
            # CNN 특징맵을 1차원 벡터로 펼침
            # [B, 32, 7, 7] -> [B, 32*7*7]
            nn.Flatten(),

            # 완전연결층
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),

            # 최종 10개 클래스 점수(logit) 출력
            # 숫자 0~9에 대한 점수임
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.net(x)


# ------------------------------------------------------------
# 12. 공통 학습/평가 함수
# ------------------------------------------------------------
def train_eval(criterion):
    # 새 모델 생성
    model = SmallCNN().to(device)

    # AdamW 옵티마이저 사용
    opt = optim.AdamW(model.parameters(), lr=2e-3)

    # 3 epoch 학습
    for _ in range(3):
        model.train()

        # 학습 DataLoader 순회
        for x, y in tr:
            x, y = x.to(device), y.to(device)

            # 이전 gradient 초기화
            opt.zero_grad()

            # 순전파 + 손실 계산
            # model(x) 출력 shape: [B, 10]
            # y shape: [B]
            #
            # CrossEntropyLoss는 raw logit을 입력받음
            # 따로 softmax를 미리 하지 않는 게 일반적임
            loss = criterion(model(x), y)

            # 역전파
            loss.backward()

            # 파라미터 업데이트
            opt.step()

    # 평가 단계
    model.eval()
    correct = 0
    tot = 0

    with torch.no_grad():
        for x, y in te:
            x, y = x.to(device), y.to(device)

            # model(x) 결과는 [B, 10] 점수
            # argmax(1)로 가장 점수가 큰 클래스 선택
            pred = model(x).argmax(1)

            # 맞춘 개수 누적
            correct += (pred == y).sum().item()

            # 전체 샘플 수 누적
            tot += y.size(0)

    # 최종 정확도 반환
    return correct / tot


# ------------------------------------------------------------
# 13. Label Smoothing 적용
# ------------------------------------------------------------
# 일반적인 CrossEntropy는 정답 클래스를 1, 나머지를 0처럼 강하게 봄
#
# label_smoothing=0.1은
# - 정답 클래스 확률을 1.0이 아니라 조금 낮추고
# - 나머지 클래스에도 아주 작은 확률을 분배하는 효과가 있음
#
# 목적
# - 모델이 지나치게 확신(over-confident)하는 걸 완화
# - 일반화 성능 개선 기대
crit_ls = nn.CrossEntropyLoss(label_smoothing=0.1)
acc_ls = train_eval(crit_ls)


# ------------------------------------------------------------
# 14. Class Weight 적용
# ------------------------------------------------------------
# 클래스별 중요도를 다르게 줄 수 있음
# 데이터 불균형이 있거나,
# 특정 클래스를 더 중요하게 다루고 싶을 때 사용함
#
# 여기서는 예시로 5, 7, 9 클래스에 1.2 가중치 부여
# 즉 이 클래스들을 틀렸을 때 손실이 조금 더 크게 반영되게 함
weights = torch.tensor(
    [1, 1, 1, 1, 1, 1.2, 1, 1.2, 1, 1.2],
    dtype=torch.float32
).to(device)

crit_w = nn.CrossEntropyLoss(weight=weights)
acc_w = train_eval(crit_w)


# ------------------------------------------------------------
# 15. 다중분류 결과 출력
# ------------------------------------------------------------
print(f"Multiclass Acc: LabelSmoothing={acc_ls:.3f} vs ClassWeight={acc_w:.3f}")


Binary Acc: MSE=0.598 vs BCEWithLogits=0.627


100%|██████████| 9.91M/9.91M [00:00<00:00, 65.3MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.77MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 15.1MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.43MB/s]


MNIST Acc: LabelSmoothing=0.990 | WeightedCE=0.986
